### - Importando bibliotecas para extrair os dados
### - yfinance é uma biblioteca que puxa os dados financeiros da b3

In [0]:
import os
import sys
import datetime
import pandas as pd
import yfinance as yf

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

### Separando empresas em segmentos

In [0]:
finance = [
'BBDC4.SA', 'BBDC3.SA',                   #Bradesco
'ITUB4.SA', 'ITUB3.SA',                   #Itau Unibanco
'ITSA4.SA', 'ITSA3.SA',                   #Itausa (holding)
'BBAS3.SA',                               #Banco do Brasil
'SANB11.SA', 'SANB4.SA', 'SANB3.SA',      #Santander Brasil
#'BPAN4.SA',                              #Banco Pan
'ABCB4.SA',                               #Banco ABC Brasil
'BMGB4.SA',                               #Banco BMG
'BRSR6.SA', 'BRSR3.SA', 'BRSR5.SA',       #Banrisul
'BMEB4.SA', 'BMEB3.SA',                   #Banco Mercantil do Brasil
'BMIN4.SA','BMIN3.SA',                    #Banco Mercantil de Investimentos
'BAZA3.SA',                               #Banco da Amazonia
'BGIP4.SA', 'BGIP3.SA',                   #Banco Banese
'BEES3.SA', 'BEES4.SA'                    #Banco Banestes
            ]

In [0]:
industria = [
'VALE3.SA',                               #Vale (mineracao)
'USIM5.SA', 'USIM3.SA',                   #Usiminas
'CSNA3.SA',                               #CSN
'GGBR4.SA', 'GGBR3.SA',                   #Gerdau
'GOAU4.SA', 'GOAU3.SA',                   #Metalurgica Gerdau
'CMIN3.SA'                                #CSN Mineracao
              ]

In [0]:
energi = [
'PETR4.SA', 'PETR3.SA',                   #Petrobras
'PRIO3.SA',                               #PetroRio
'RAIZ4.SA',                               #Raizen
'UGPA3.SA',                               #Ultrapar
'ENEV3.SA',                               #Eneva
'EQTL3.SA',                               #Equatorial Energia
'CMIG4.SA', 'CMIG3.SA'                    #Cemig
#'ELET3.SA', 'ELET6.SA', 'ELET5.SA'        #Eletrobras
           ]

In [0]:
varejo = [
'MGLU3.SA',                               #Magazine Luiza
'LREN3.SA',                               #Lojas Renner
#'PETZ3.SA',                              #Petz
'AMER3.SA',                               #Americanas
'ASAI3.SA',                               #Assai
'PCAR3.SA',                               #Pao de Acucar
'VAMO3.SA',                               #Vamos Locadora
'RENT3.SA'                                #Localiza
           ]

In [0]:
tecnologia = [
'TOTS3.SA',                                #Totvs
'LWSA3.SA',                                #Locaweb
'YDUQ3.SA',                                #Yduqs
'COGN3.SA',                                #Cogna
'ANIM3.SA'                                 #Anima Educacao
               ]

In [0]:
food = [
'ABEV3.SA',                                #Ambev
#'BRFS3.SA',                               #BRF
#'MRFG3.SA',                               #Marfrig
'BEEF3.SA'                                 #Minerva
         ]

### Trazendo dados no perido de 10 anos

In [0]:
empresas = finance + industria + energi + varejo + tecnologia + food

hoje = datetime.datetime.today().strftime('%Y-%m-%d')

inicio = (datetime.datetime.today() - datetime.timedelta(days=365*10)).strftime('%Y-%m-%d')

dados = yf.download(empresas, start=inicio, end=hoje)

[*********************100%***********************]  56 of 56 completed


### Criando setores de empresas

In [0]:
dados_setores = []

for ticker_str in empresas:
    try:
        empresas = yf.Ticker(ticker_str)
        info = empresas.info

        dados_setores.append({
            'Codigo': ticker_str,
            'Setor': info.get('sector', 'Indefinido'),
            'Industria': info.get('industry', 'Indefinida'),
            'Nome_Empresa': info.get('longName', 'Desconhecido')
        })

    except Exception as e:
        print(f"Erro ao buscar {ticker_str}: {e}")

In [0]:
dados_empilhado = dados.stack(level=1).reset_index()

/home/spark-e2c48ac6-4a8e-42c6-ad4f-4d/.ipykernel/19019/command-6276982127560247-1759546269:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  dados_empilhado = dados.stack(level=1).reset_index()


### Importando biblioteca sobre SKPARK

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import date_format
from pyspark.sql.window import Window

In [0]:
df_dados = spark.createDataFrame(dados_empilhado)

In [0]:
df_dados.count()

132886

In [0]:
display(df_dados.limit(100))

Date,Ticker,Close,High,Low,Open,Volume
2016-06-14T00:00:00.000Z,ABCB4.SA,5.471071243286133,5.641898127986096,5.4664541466838905,5.614196022504559,143325.0
2016-06-14T00:00:00.000Z,ABEV3.SA,12.410421371459961,12.624278728313554,12.310175894771138,12.597547196052053,1.61056E7
2016-06-14T00:00:00.000Z,AMER3.SA,853.5361938476562,861.9777221356997,843.2187105511426,851.6602986725355,8990.0
2016-06-14T00:00:00.000Z,ANIM3.SA,3.1796467304229736,3.2657339657582307,3.1352154162758286,3.2129707125968765,366300.0
2016-06-14T00:00:00.000Z,BAZA3.SA,12.024175643920898,12.024175643920898,12.024175643920898,12.024175643920898,1604.0
2016-06-14T00:00:00.000Z,BBAS3.SA,4.37103796005249,4.566391418193392,4.3140595446507115,4.512126309954596,1.76298E7
2016-06-14T00:00:00.000Z,BBDC3.SA,7.317416191101074,7.514635968500407,7.2130064753667495,7.41892652786434,3894792.0
2016-06-14T00:00:00.000Z,BBDC4.SA,7.055229663848877,7.30257481827945,6.928612333479634,7.175957487910209,2.3603763E7
2016-06-14T00:00:00.000Z,BEEF3.SA,6.865180492401123,7.008361891603916,6.842572710714757,6.978217939059923,379845.0
2016-06-14T00:00:00.000Z,BEES3.SA,1.0172516107559204,1.0172516107559204,0.9324807699547542,0.9726353479629599,5720.0


In [0]:
df_setores = spark.createDataFrame(dados_setores)

In [0]:
df_setores.count()

56

In [0]:
display(df_setores.limit(100))

Codigo,Industria,Nome_Empresa,Setor
BBDC4.SA,Banks - Regional,Banco Bradesco S.A.,Financial Services
BBDC3.SA,Banks - Regional,Banco Bradesco S.A.,Financial Services
ITUB4.SA,Banks - Regional,Itaú Unibanco Holding S.A.,Financial Services
ITUB3.SA,Banks - Regional,Itaú Unibanco Holding S.A.,Financial Services
ITSA4.SA,Conglomerates,Itaúsa S.A.,Industrials
ITSA3.SA,Conglomerates,Itaúsa S.A.,Industrials
BBAS3.SA,Banks - Regional,Banco do Brasil S.A.,Financial Services
SANB11.SA,Banks - Regional,Banco Santander (Brasil) S.A.,Financial Services
SANB4.SA,Banks - Regional,Banco Santander (Brasil) S.A.,Financial Services
SANB3.SA,Banks - Regional,Banco Santander (Brasil) S.A.,Financial Services


### Mudando nome dos setores

In [0]:
df_setores = df_setores.withColumn(
    "setor_atuacao",
    F.when(F.col("Setor") == "Financial Services",
           "Financeiro")
    .when(F.col("Setor") == "Health Care",
          "Saude")
    .when(F.col("Setor") == "Consumer Defensive",
          "Consumo Defensivo")
    .when(F.col("Setor") == "Consumer Cyclical",
          "Consumo Cíclico")
    .when(F.col("Setor") == "Real Estate",
          "Imobiliario")
    .when(F.col("Setor") == "Basic Materials",
          "Materiais Basicos")
    .when(F.col("Setor") == "Energy",
          "Energia")
    .when(F.col("Setor") == "Industrials",
          "Industrial")
    .when(F.col("Setor") == "Consumer Discretionary",
          "Consumo Discricionario")
    .when(F.col("Setor") == "Consumer Staples",
          "Consumo Essencial")
    .when(F.col("Setor") == "Utilities",
          "Servicos Publicos")
    .when(F.col("Setor") == "Communication Services",
          "Servicos de Comunicacao")
    .when(F.col("Setor") == "Healthcare",
          "Saude")
    .when(F.col("Setor") == "Technology",
          "Tecnologia")
    .when(F.col("Setor") == "Fixtures & Accessories",
          "Acessórios e Utensílios")
    .when(F.col("Setor") == "Consumer Services",
          "Servicos de Consumo")
    .otherwise(F.col("Setor"))
)

### Mudando nome dos segmentos

In [0]:
df_setores = df_setores.withColumn(
    "segmento",
    F.when(F.col("Industria") == "Financial Data & Stock Exchanges",
           "Dados Financeiros e Bolsas de Valores")
    .when(F.col("Industria") == "Real Estate - Development",
          "Imobiliario - Incorporacao")
    .when(F.col("Industria") == "Other Industrial Metals & Mining",
          "Outros Metais Industriais e Mineracao")
    .when(F.col("Industria") == "Banks - Regional",
          "Bancos Regionais")
    .when(F.col("Industria") == "Insurance - Life",
          "Seguros de Vida")
    .when(F.col("Industria") == "Oil & Gas Integrated",
          "Petroleo e Gas - Integrado")
    .when(F.col("Industria") == "Airlines",
          "Companhias Aereas")
    .when(F.col("Industria") == "Steel",
          "Aco")
    .when(F.col("Industria") == "Education & Training Services",
          "Educacao e Servicos de Treinamento")
    .when(F.col("Industria") == "Department Stores",
          "Lojas de Departamento")
    .when(F.col("Industria") == "Beverages - Brewers",
          "Bebidas - Cervejarias")
    .when(F.col("Industria") == "Grocery Stores",
          "Supermercados")
    .when(F.col("Industria") == "Railroads",
          "Ferrovias")
    .when(F.col("Industria") == "Conglomerates",
          "Conglomerados")
    .when(F.col("Industria") == "Oil & Gas E&P",
          "Exploracao e Producao de Petroleo e Gas")
    .when(F.col("Industria") == "Utilities - Renewable",
          "Utilidades - Energias Renovaveis")
    .when(F.col("Industria") == "Specialty Retail",
          "Varejo Especializado")
    .when(F.col("Industria") == "Specialty Business Services",
          "Servicos Empresariais Especializados")
    .when(F.col("Industria") == "Utilities - Diversified",
          "Utilidades - Diversificadas")
    .when(F.col("Industria") == "Farm & Heavy Construction Machinery",
          "Maquinas Agricolas e de Construcao Pesada")
    .when(F.col("Industria") == "Travel Services",
          "Servicos de Viagem")
    .when(F.col("Industria") == "Oil & Gas Refining & Marketing",
          "Refino e Comercializacao de Petroleo e Gas")
    .when(F.col("Industria") == "Marine Shipping",
          "Transporte Maritimo")
    .when(F.col("Industria") == "Aluminum",
          "Aluminio")
    .when(F.col("Industria") == "Utilities - Regulated Electric",
          "Utilidades - Eletricidade Regulada")
    .when(F.col("Industria") == "Electrical Equipment & Parts",
          "Equipamentos e Pecas Eletricas")
    .when(F.col("Industria") == "Rental & Leasing Services",
          "Servicos de Aluguel e Arrendamento")
    .when(F.col("Industria") == "Packaged Foods",
          "Alimentos Embalados")
    .when(F.col("Industria") == "Capital Markets",
          "Mercados de Capitais")
    .when(F.col("Industria") == "Insurance - Diversified",
          "Seguros - Diversificados")
    .when(F.col("Industria") == "Residential Construction",
          "Construcao Residencial")
    .when(F.col("Industria") == "Telecom Services",
          "Servicos de Telecomunicacoes")
    .when(F.col("Industria") == "Healthcare Plans",
          "Planos de Saude")
    .when(F.col("Industria") == "Farm Products",
          "Produtos Agricolas")
    .when(F.col("Industria") == "Paper & Paper Products",
          "Papel e Produtos de Papel")
    .when(F.col("Industria") == "Asset Management",
          "Gestao de Ativos")
    .when(F.col("Industria") == "Infrastructure Operations",
          "Operacoes de Infraestrutura")
    .when(F.col("Industria") == "Household & Personal Products",
          "Produtos Domesticos e Pessoais")
    .when(F.col("Industria") == "Medical Care Facilities",
          "Instalacoes de Cuidados Medicos")
    .when(F.col("Industria") == "Aerospace & Defense",
          "Aeroespacial e Defesa")
    .when(F.col("Industria") == "Lumber & Wood Production",
          "Producao de Madeira e Serraria")
    .when(F.col("Industria") == "Pharmaceutical Retailers",
          "Farmecias e Varejo Farmaceutico")
    .when(F.col("Industria") == "Software - Infrastructure",
          "Software - Infraestrutura")
    .when(F.col("Industria") == "Real Estate Services",
          "Servicos Imobiliarios")
    .when(F.col("Industria") == "Electronics & Computer Distribution",
          "Distribuição de Eletronicos e Computadores")
    .when(F.col("Industria") == "Software - Application",
          "Software - Aplicacoes")
    .when(F.col("Industria") == "Utilities - Regulated Water",
          "Utilidades - Agua Regulada")
    .when(F.col("Industria") == "Chemicals",
          "Produtos Quimicos")
    .when(F.col("Industria") == "Footwear & Accessories",
          "Calcados e Acessorios")
    .when(F.col("Industria") == "Diagnostics & Research",
          "Diagnostico e Pesquisa")
    .when(F.col("Industria") == "Luxury Goods",
          "Bens de Luxo")
    .when(F.col("Industria") == "Internet Retail",
          "Varejo Online")
    .when(F.col("Industria") == "Medical Distribution ",
          "Distribuicao Medica")
    .when(F.col("Industria") == "Communication Equipment",
          "Equipamentos de Comunicacao")
    .when(F.col("Industria") == "Drug Manufacturers - Specialty & Generic",
          "Fabricantes de Medicamentos - Especializados e Genéricos")
    .when(F.col("Industria") == "Integrated Freight & Logistics",
          "Transporte e Logistica Integrada")
    .when(F.col("Industria") == "Engineering & Construction",
          "Engenharia e Construcao")
    .when(F.col("Industria") == "Specialty Chemicals",
          "Produtos Químicos Especiais")
    .when(F.col("Industria") == "Internet Content & Information",
          "Conteudo e Informacao na Internet")
    .when(F.col("Industria") == "Leisure",
          "Lazer")
    .when(F.col("Industria") == "Computer Hardware",
          "Hardware de Computadores")
    .when(F.col("Industria") == "Specialty Industrial Machinery",
          "Maquinas Industriais Especiais")
    .when(F.col("Industria") == "Waste Management",
          "Gerenciamento de Residuos")
    .when(F.col("Industria") == "Auto Parts",
          "Pecas Automotivas")
    .when(F.col("Industria") == "Confectioners",
          "Confeitaria")
    .when(F.col("Industria") == "Restaurants",
          "Restaurantes")
    .when(F.col("Industria") == "Packaging & Containers ",
          "Embalagens e Recipientes")
    .when(F.col("Industria") == "Insurance - Reinsurance",
          "Resseguros")
    .when(F.col("Industria") == "Agricultural Inputs",
          "Insumos Agricolas")
    .when(F.col("Industria") == "Apparel Retail",
          "Varejo de Roupas")
    .when(F.col("Industria") == "Building Products & Equipment ",
          "Produtos e Equipamentos para Construcao")
    .when(F.col("Industria") == "Real Estate - Diversified",
          "Imobiliario - Diversificado")
    .when(F.col("Industria") == "Metal Fabrication",
          "Fabricacao de Metais")
    .when(F.col("Industria") == "Drug Manufacturers - General",
          "Fabricantes de Medicamentos - Gerais")
    .when(F.col("Industria") == "Insurance Brokers",
          "Corretores de Seguros")
    .when(F.col("Industria") == "Entertainment",
          "Entretenimento")
    .when(F.col("Industria") == "Industrial Distribution",
          "Distribuicao Industrial")
    .when(F.col("Industria") == "Copper",
          "Cobre")
    .when(F.col("Industria") == "Trucking",
          "Transporte Rodoviario")
    .when(F.col("Industria") == "Personal Services",
          "Servicos Pessoais")
    .when(F.col("Industria") == "Advertising Agencies",
          "Agencias de Publicidade")
    .when(F.col("Industria") == "Textile Manufacturing",
          "Industria Textil")
    .when(F.col("Industria") == "Oil & Gas Equipment & Services",
          "Equipamentos e Servicos para Petroleo e Gas")
    .when(F.col("Industria") == "Biotechnology",
          "Biotecnologia")
    .when(F.col("Industria") == "Furnishings",
          "Mobiliario e Decoracao")
    .when(F.col("Industria") == "Apparel Manufacturing",
          "Industria de Vestuario")
    .when(F.col("Industria") == "Utilities - Regulated Gas",
          "Utilidades - Gas Regulamentado")
    .when(F.col("Industria") == "Auto & Truck Dealerships",
          "Concessionarias de Automoveis e Caminhoes")
    .when(F.col("Industria") == "Lodging",
          "Hospedagem")
    .when(F.col("Industria") == "Medical Instruments & Supplies",
          "Instrumentos e Suprimentos Medicos")
    .when(F.col("Industria") == "Shell Companies ",
          "Empresas de Proposito Especifico (Shell Companies)")
    .otherwise(F.col("Industria"))
    )

### Juntando informacoes entre os dados e o de para

In [0]:
df_acoes = df_dados.alias('dados').join(
    df_setores.alias('setor'),
    F.col("dados.Ticker") == F.col('setor.codigo'),
    "inner")

In [0]:
tb_acoes = df_acoes.select(
    F.col("dados.Date").alias("data"),
    F.col("setor.Nome_Empresa").alias("empresa"),
    F.col("setor.setor_atuacao").alias("setor"),
    F.col("setor.segmento").alias("segmento"),
    F.col("dados.Ticker").alias("cd_empresa"),
    F.col("dados.Open").cast("decimal(38,2)").alias("abertura"),
    F.col("dados.High").cast("decimal(38,2)").alias("maxima"),
    F.col("dados.Low").cast("decimal(38,2)").alias("minima"),
    F.col("dados.Close").cast("decimal(38,2)").alias("fechamento"),
    F.col("dados.Volume").cast("bigint").alias("volume"))

In [0]:
tb_acoes = tb_acoes.withColumn("abertura",
                               F.regexp_replace(F.col("abertura"),
                                                "\\.", ",")
                               )

tb_acoes = tb_acoes.withColumn("maxima",
                               F.regexp_replace(F.col("maxima"),
                                                "\\.", ",")
                               )

tb_acoes = tb_acoes.withColumn("minima",
                               F.regexp_replace(F.col("minima"),
                                                "\\.", ",")
                               )

tb_acoes = tb_acoes.withColumn("fechamento",
                               F.regexp_replace(F.col("fechamento"),
                                                "\\.", ",")
                               )

In [0]:
display(tb_acoes.limit(100))

data,empresa,setor,segmento,cd_empresa,abertura,maxima,minima,fechamento,volume
2016-06-14T00:00:00.000Z,Banco ABC Brasil S.A.,Financeiro,Bancos Regionais,ABCB4.SA,"5,61","5,64","5,47","5,47",143325
2016-06-14T00:00:00.000Z,Ambev S.A.,Consumo Defensivo,Bebidas - Cervejarias,ABEV3.SA,"12,60","12,62","12,31","12,41",16105600
2016-06-14T00:00:00.000Z,Americanas S.A.,Consumo Cíclico,Varejo Online,AMER3.SA,"851,66","861,98","843,22","853,54",8990
2016-06-14T00:00:00.000Z,Ânima Holding S.A.,Consumo Defensivo,Educacao e Servicos de Treinamento,ANIM3.SA,"3,21","3,27","3,14","3,18",366300
2016-06-14T00:00:00.000Z,Banco da Amazônia S.A.,Financeiro,Bancos Regionais,BAZA3.SA,"12,02","12,02","12,02","12,02",1604
2016-06-14T00:00:00.000Z,Banco do Brasil S.A.,Financeiro,Bancos Regionais,BBAS3.SA,"4,51","4,57","4,31","4,37",17629800
2016-06-14T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC3.SA,"7,42","7,51","7,21","7,32",3894792
2016-06-14T00:00:00.000Z,Banco Bradesco S.A.,Financeiro,Bancos Regionais,BBDC4.SA,"7,18","7,30","6,93","7,06",23603763
2016-06-14T00:00:00.000Z,Minerva S.A.,Consumo Defensivo,Alimentos Embalados,BEEF3.SA,"6,98","7,01","6,84","6,87",379845
2016-06-14T00:00:00.000Z,Banestes S.A - Banco do Estado do Espírito Santo,Financeiro,Bancos Regionais,BEES3.SA,"0,97","1,02","0,93","1,02",5720
